# Dataset Cleaning Pipeline

In [1]:
# Imports
import sys, os
sys.path.insert(0, '../src')
from skrub import TableReport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import preprocess
import explore
from preprocess import PRE_EXCLUDED

In [2]:
# Load datasets
df_im, df_cl = explore.load_data()

# Immunological Dataset

In [3]:
explore.dataset_overview(df_im, name="Immunological Dataset")


Raw Immunological Dataset Dataset Overview
  Shape         : 829 rows × 137 columns
  Patients      : 264
  Timepoints    : 5

Rows per timepoint:
Timepoint
1.0    250
2.0    214
3.0    141
4.0    136
5.0     82

Missing values: 1115 (1.0% of all cells)

Column dtypes:
object: 69 columns
float64: 68 columns




In [2]:
# Censored from Gitlab
# #print('TableReport of Raw Immunological Dataset')
#TableReport(df_im, max_plot_columns=139)

## 1:   Drop Pre-Determined Columns and Patients

In [4]:
# 1: Drop pre-determined columns and patients, and rename column
df_im_vis = df_im.copy()
df_im_vis = preprocess.drop_rename_cols_im(df_im_vis, verbose=True)
df_im_vis = df_im_vis[~df_im_vis['Patient'].isin(PRE_EXCLUDED)]
print('Unique patients after dropping pre-determined patients (264 before):', df_im_vis['Patient'].nunique())
print('Flagged patients for exclusion in clinical dataset that were not found in immunolgical dataset:', set(PRE_EXCLUDED) - set(df_im['Patient'].unique()))

  Dropped 45 columns
Unique patients after dropping pre-determined patients (264 before): 228
Flagged patients for exclusion in clinical dataset that were not found in immunolgical dataset: {265, 28, 57}


## 2: Replace Missing-Value Markers with NaN

In [5]:
# 2: Replacing missing-value markers with NaN
df_im_vis = preprocess.replace_missing_markers(df_im_vis, skip_cols=["Patient", "Timepoint"], verbose=True)

  Neu_CD25+: replaced 144 null markers
  BAS_CD25+: replaced 18 null markers
  Mo_CD25+: replaced 97 null markers
  T_CD25hi: replaced 5 null markers
  TC_CD25hi: replaced 726 null markers
  TH_CD25hi: replaced 5 null markers
  B_CD25+: replaced 7 null markers
  B_CD25hi: replaced 745 null markers
  NK_CD25+: replaced 134 null markers
  Neu_HLADR+: replaced 25 null markers
  Eos_HLADR+: replaced 522 null markers
  Mo_HLADRhi: replaced 4 null markers
  Mo2_HLADR+: replaced 173 null markers
  Mo3_HLADR+: replaced 68 null markers
  Mo1_HLADRhi: replaced 11 null markers
  Mo2_HLADRhi: replaced 210 null markers
  Mo3_HLADRhi: replaced 85 null markers
  TC_HLADR+: replaced 7 null markers
  T_HLADRhi: replaced 12 null markers
  TC_HLADRhi: replaced 695 null markers
  TH_HLADRhi: replaced 35 null markers
  B_HLADR+: replaced 1 null markers
  NK_HLADR+: replaced 48 null markers
  NK_HLADRhi: replaced 588 null markers
  Neu_CD69+: replaced 36 null markers
  Eos_CD69+: replaced 350 null markers
 

## 4: Drop Empty Rows + Missing Data Rows

In [6]:
# 3: Drop known empty rows + all-NaN feature rows
df_im_vis = preprocess.drop_empty_rows_im(df_im_vis, verbose=True)

Verifying found empty rows:
  Row 823  Patient=nan  OK — all NaN
  Row 824  Patient=nan  OK — all NaN
  Row 825  Patient=nan  OK — all NaN
  Row 826  Patient=nan  OK — all NaN
  Row 827  Patient=nan  OK — all NaN
  Row 828  Patient=nan  WARNING — 1 non-NaN values:
Eosinophils    All yellow columns can be excluded from the an...

Dropped 6

Dropping 1 all-NaN feature rows:
    Patient  Timepoint
72     30.0        2.0

Unique patients remaining: 228


## 5: Convert column dtypes

In [7]:
# 4: Correct dtypes of columns
df_im_vis = preprocess.fix_dtypes_im(df_im_vis, verbose=True)



Data types after cleaning:
float64           89
Int64              2
datetime64[ns]     1
Name: count, dtype: int64


### Dataset Ready for Visualisation

In [8]:
explore.dataset_overview(df_im_vis, name="Immunological dataset after cleaning for visualization")


Raw Immunological dataset after cleaning for visualization Dataset Overview
  Shape         : 757 rows × 92 columns
  Patients      : 228
  Timepoints    : 5

Rows per timepoint:
Timepoint
1    214
2    201
3    135
4    131
5     76

Missing values: 8293 (11.9% of all cells)

Column dtypes:
float64: 89 columns
Int64: 2 columns
datetime64[ns]: 1 columns




In [3]:
# Censored from Gitlab
#TableReport(df_im_vis, max_plot_columns=100)

## 5:  Removing >25% NaN Columns and Found Outliers

In [ ]:
# 6: Remove columns with more than 25% nan 
df_im_mod = preprocess.remove_nan_cols(df_im_vis, threshold=0.25, verbose=True)  # copy used for modeling 

# 7: Remove found outlier observations in dataset:
df_im_mod = preprocess.remove_outlier_observations(df_im_mod, verbose=True)
# Note: patient 150 T1 has been removed previously (exclusion criteria)

  Dropped 14 columns with >25% NaN: ['TC_CD25hi', 'B_CD25hi', 'Eos_HLADR+', 'Mo2_HLADRhi', 'TC_HLADRhi', 'NK_HLADRhi', 'Eos_CD69+', 'Bas_CD69+', 'Mo_CD69+', 'B_CD69+', 'DC_CD69+', 'TH naive_PD1+', 'TH eff_PD1+', 'TC naive_PD1+']
  Removed 7 observations:
    Patient 221  T2  (removed)
    Patient 163  T1  (removed)
    Patient 150  T1  (not found)
    Patient 159  T2  (removed)
    Patient 109  T5  (removed)
    Patient 266  T4  (removed)
    Patient 254  T1  (removed)
    Patient 229  T2  (removed)
  Shape before: (757, 78)  Shape after: (750, 78)
  Rows removed: 7


## 6: Dropping Unused Columns for Modeling

In [11]:
df_im_mod = preprocess.remove_for_modeling(df_im_mod, verbose=True)

  Dropped 1 columns: ['Date']


### Dataset Ready for Modeling

In [12]:
explore.dataset_overview(df_im_mod, name="Immunological dataset after cleaning for modeling")


Raw Immunological dataset after cleaning for modeling Dataset Overview
  Shape         : 750 rows × 77 columns
  Patients      : 227
  Timepoints    : 5

Rows per timepoint:
Timepoint
1    212
2    198
3    135
4    130
5     75

Missing values: 1316 (2.3% of all cells)

Column dtypes:
float64: 75 columns
Int64: 2 columns




In [4]:
# Censored from Gitlab
#TableReport(df_im_mod, max_plot_columns=100)

# Clinical Dataset

In [14]:
explore.dataset_overview(df_cl, name="Raw Clinical Dataset")


Raw Raw Clinical Dataset Dataset Overview
  Shape         : 1657 rows × 79 columns
  Patients      : 276

Missing values: 98132 (75.0% of all cells)

Column dtypes:
object: 67 columns
float64: 12 columns




In [5]:
print('TableReport of Raw Clinical Dataset')
# Censored from Gitlab
#TableReport(df_cl, max_plot_columns=130)

TableReport of Raw Clinical Dataset


## 1: Construct Timepoint Column + Forward-Fill Clinical Variables

In [16]:
# 1: Forward-fill Clinical Variables and construct Timepoint Column
df_cl_vis = df_cl.copy() 
df_cl_vis = preprocess.forward_fill_clinical(df_cl_vis, verbose=True)

  df_cl initialised: 1657 rows × 80 columns


## 2: Exclude Pre-Determined Patients and Columns

In [17]:
df_cl_vis = preprocess.exclude_predetermined(df_cl_vis, verbose=True)

  Excluded 28 patients by Ausschluss keyword: [  2.  13.  26.  27.  28.  37.  39.  40.  41.  44.  46.  50.  57.  67.
  68.  71.  74.  75.  79.  85.  96.  98. 100. 168. 216. 260. 264. 265.]

  Verifying multi-body-part patients:
    Patient 3: Target volume(s) = ['heel L + thumb L']
    Patient 45: Target volume(s) = ['wrist with thumb joint']
    Patient 184: Target volume(s) = ['Hip L& R and Knie L&R']
    Patient 149: Target volume(s) = ['heel R, Ankle R']
    Patient 150: Target volume(s) = ['heel L, ankle L']
    Patient 162: Target volume(s) = ['heel L, Knie L']
    Patient 179: Target volume(s) = ['heel L,R;\ntoe R']
    Patient 156: Target volume(s) = ['knee L& R heel L']
    Patient 54: Target volume(s) = ['ankle joint R + midfoot L']
    Patient 47: Target volume(s) = ['thumb-saddle-joint L + forefoot R']
  Removed 10 multi-body-part patients

  Dropped 48 questionnaire columns ('Schwierigkeiten körperlicher Anstrengung' to 'Allgemeinzustand Gesundheut HEUTE')

  Dropped 4 emp

## 3: Rename Columns + Manual Corrections

In [18]:
# 3: Renaming columns from German to English
df_cl_vis = preprocess.rename_columns_cl(df_cl_vis, verbose=True)

# Manual corretions of typos in dataset
df_cl_vis = preprocess.manual_corrections_cl(df_cl_vis, verbose=True)


  Renamed 26 columns
  Patient 248 T2 pain_daytime set to '2' (was '22')
  Removed Patient 219 (different questionnaire): 6 rows dropped
  Patient 89 row 27.03.2019 → Timepoint 2
  Patient 89 row 05.07.2019 → Timepoint 5
  Removed Patient 89 row dated 10.05.2019 (unmatched timepoint)


## 4:  Parsing, Splitting and Transforming Columns

In [19]:
# 4: Parsing, splitting and transforming columns
df_cl_vis = preprocess.parse_transform_cl(df_cl_vis, verbose=True)

Parsing and Transforming Clinical columns: 
Unique Value Counts before and after

--- diagnosis (BEFORE, 98 unique values) ---
diagnosis
Calcaneodynie L                               32
Calcaneodynie R                               25
NaN                                           15
Calcaneodynie L,R                             10
ellbow syndrom L                               8
heel calcaneodynia L                           7
Achillodynie L                                 6
ellbow syndrom R                               5
shouldersyndrom R                              5
achillodynia R                                 4
achillodynia L                                 4
calcaneodynia L + R                            4
Rizarthrosis R                                 3
gonarthrosis both sides (knee)                 3
n.D                                            3
heel calcaneodynia R                           3
Elbow syndrome R                               3
thumb CMC joint arthritis L + 

## 5: Replace Missing-Value Markers + Drop Empty Rows

In [20]:
# Repleace missing-value markers with NaN
df_cl_vis = preprocess.replace_missing_markers(df_cl_vis, skip_cols=["Patient", "Timepoint"], verbose=True)
print("\n")
# Dropping empty rows and rows with no pain-related data
df_cl_vis = preprocess.drop_rows_cl(df_cl_vis, verbose=True)

  weight_kg: replaced 108 null markers
  height_cm: replaced 108 null markers
  date: replaced 39 null markers
  complaints_since: replaced 60 null markers
  pain_points: replaced 19 null markers
  diagnosis: replaced 18 null markers
  target_volume: replaced 18 null markers
  single_fraction: replaced 36 null markers
  fha: replaced 96 null markers
  kv: replaced 96 null markers
  ma: replaced 96 null markers
  filter: replaced 96 null markers
  response: replaced 114 null markers


  Dropping 579 empty rows:

After dropping empty rows: 233 patients, 843 rows


## 6: Correct Column dtypes

In [21]:
df_cl_vis = preprocess.fix_dtypes_cl(df_cl_vis, verbose=True)

  Dropping rows with unknown Patient or Timepoint:
      Patient  Timepoint
1082    181.0        NaN

--- Dtype summary (clinical dataset) ---
float64           24
int64              2
category           1
category           1
object             1
datetime64[ns]     1
category           1
category           1
category           1
category           1
category           1
category           1
Name: count, dtype: int64
Shape: (842, 36), Patients: 233


### Dataset Ready for Visualisation

In [ ]:
explore.dataset_overview(df_cl_vis, name="Clinical Dataset after cleaning for visualization")


Raw Clinical Dataset after cleaning for visualization Dataset Overview
  Shape         : 842 rows × 36 columns
  Patients      : 233
  Timepoints    : 5

Rows per timepoint:
Timepoint
1    193
2    210
3    165
4    152
5    122

Missing values: 3069 (10.1% of all cells)

Column dtypes:
float64: 24 columns
int64: 2 columns
category: 1 columns
category: 1 columns
object: 1 columns
datetime64[ns]: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns




In [6]:
# Censored from Gitlab
#TableReport(df_cl_vis, max_plot_columns=100)

##  7: Dropping >25% nan Columns 

In [24]:
# Remove columns with more than 25% nan 
df_cl_mod = preprocess.remove_nan_cols(df_cl_vis, threshold=0.25, verbose=True)  # copy used for modeling 

  Dropped 3 columns with >25% NaN: ['complaints_since', 'improvement_percent', 'filter']


## 8: Dropping Outliers and Unused Columns for Modeling

In [ ]:
df_cl_mod = preprocess.remove_for_modeling(df_cl_mod, verbose=True)
df_cl_mod = preprocess.remove_outlier_observations(df_cl_mod)
# note: patient 150 t1 has been removed previously (exclusion criteria), outlier patient 226 T4 was not present in clinical dataset
from model import cl_leaky_columns
print(f"Leaky Columns that will be ignored during modeling:{cl_leaky_columns}")


  Dropped 7 columns: ['kv', 'ma', 'fha', 'single_fraction', 'pain_points', 'date', 'measurement_timepoint']
  Removed 6 observations:
    Patient 221  T2  (removed)
    Patient 163  T1  (removed)
    Patient 150  T1  (not found)
    Patient 159  T2  (removed)
    Patient 109  T5  (removed)
    Patient 266  T4  (not found)
    Patient 254  T1  (removed)
    Patient 229  T2  (removed)
  Shape before: (842, 26)  Shape after: (836, 26)
  Rows removed: 6
Leaky Columns that will be ignored during modeling:['response', 'improvement_percent', 'pain_scale', 'pain_under_load', 'pain_night', 'pain_daytime', 'pain_at_rest', 'morning_stiffness']


### Dataset Ready for Modeling

In [26]:
explore.dataset_overview(df_cl_mod, name="Clinical Dataset after cleaning for modeling")


Raw Clinical Dataset after cleaning for modeling Dataset Overview
  Shape         : 836 rows × 26 columns
  Patients      : 233
  Timepoints    : 5

Rows per timepoint:
Timepoint
1    191
2    207
3    165
4    152
5    121

Missing values: 1583 (7.3% of all cells)

Column dtypes:
float64: 18 columns
int64: 2 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns
category: 1 columns




In [7]:
# Censored from Gitlab
#TableReport(df_cl_mod, max_plot_columns=100)

## Common patients in clinical and immunological dataset:

In [28]:
# number of unique patients in both the immunological and clinical datasets at the same time after cleaning (viz dataset)
patients_im = set(df_im_vis['Patient'].unique())
patients_cl = set(df_cl_vis['Patient'].unique())
# unique patients present in both datasets
patients_both = patients_im.intersection(patients_cl)
print('Unique patients in clinical dataset after preprocessing for visualization:', len(patients_cl))
print('Unique patients in immunological dataset after preprocessing for visualization:', len(patients_im))
print('Patients present in both datasets:', len(patients_both))

# After preprocessing for modeling:
patients_im_mod = set(df_im_mod['Patient'].unique())
patients_cl_mod = set(df_cl_mod['Patient'].unique())
# unique patients present in both datasets
patients_both_mod = patients_im_mod.intersection(patients_cl_mod)
print('\n')
print('Unique patients in clinical dataset after preprocessing for modeling:', len(patients_cl_mod))
print('Unique patients in immunological dataset after preprocessing for modeling:', len(patients_im_mod))
print('Patients present in both datasets:', len(patients_both_mod))

Unique patients in clinical dataset after preprocessing for visualization: 233
Unique patients in immunological dataset after preprocessing for visualization: 228
Patients present in both datasets: 225


Unique patients in clinical dataset after preprocessing for modeling: 233
Unique patients in immunological dataset after preprocessing for modeling: 227
Patients present in both datasets: 224
